## We aim in this notebook to identify duplicates in a CSV containing information about restaurants

In [2]:
import pandas as pd
import nltk
import time
import numpy as np

In [57]:
df_restaurants = pd.read_csv("restaurants.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [58]:
df_restaurants.head(5)

,name,address,city,cuisine,unique_id
0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,21 club,21 w. 52nd st.,new york,american,23
3,2223,2223 market st.,san francisco,american,453
4,9 jones street,9 jones st.,new york,american,173


The dataframe contains duplicates records that represent to the same 'real-world' restanrants. The column 'unique_id' was added for this purpose. Two records that are associated with the same attribute value for unique_id represents the same restaurant.

In [ ]:
df_restaurants[df_restaurants.unique_id == '23']

,name,address,city,cuisine,unique_id
2,21 club,21 w. 52nd st.,new york,american,23
753,21 club,21 w. 52nd st.,new york city,american (new),23


In the above example, the two records share the same value for attributes 'name' and 'address'. However, they have slightly different values for the columns 'city' and 'cuisine'

In [ ]:
df_restaurants[df_restaurants.unique_id == '22']

,name,address,city,cuisine,unique_id
744,yujean kangs gourmet chinese cuisine,67 n. raymond ave.,los angeles,asian,22
864,yujean kangs,67 n. raymond ave.,pasadena,chinese,22


In the above example, on the other hand, the two records are associated with different names, cities and cuisiones.

This file represents a simple example of datasets, on which we can experiment with th etechniques presented in the course to try identify duplicates, without using (that is relying on the values of) the column "unique_id".

In [59]:
# We start by adding a new column to identify the records (lines) in our dataframe
df_restaurants.insert(0,'record_ID', range(0, len(df_restaurants)))

In [60]:
df_restaurants.head(5)

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173


In [61]:
df_restaurants.iloc[0, 2]

' 103 w. paces ferry rd.'

# Exhaustive comparisons: every record is compared with every other record

We start by applying an exhaustive strategy whereby every record in the CSV file, is compared with every other record.

The code below does this for us. In doing so, it uses the following rule:

For two records to match, i.e. refer to the same restaurant in the real world:
* The edit distance between the attribute name values of the two records needs to be smaller or equal to 2, and
* they need to have the same value for the cuisine attribute.


In [102]:
import time
import nltk
from nltk.tokenize import word_tokenize
# Constants
MAX_NAME_DISTANCE = 0.5  # Adjust as needed for your dataset
MAX_ADDRESS_DISTANCE = 0.2  # Adjust as needed for your dataset
MAX_CITY_DISTANCE = 0.6 # Adjust as needed for your dataset
# Function to calculate Jaccard similarity
def jaccard_similarity(str1, str2):
    set1 = set(str1.lower().split())
    set2 = set(str2.lower().split())
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union != 0 else 0
# Function to preprocess and tokenize text

def detect_matches(df):
    num_records = len(df)
    matches = []
    number_of_matches = 0
    start = time.process_time()
    for i in range(num_records):
        for j in range(i+1, num_records):
            name_tokens1 = df.iloc[i, 1]
            name_tokens2 = df.iloc[j, 1]
            address_tokens1 = df.iloc[i, 2]
            address_tokens2 = df.iloc[j, 2]
            city_tokens1 = df.iloc[i, 3]
            city_tokens2 = df.iloc[j, 3]
            # Calculate Jaccard similarity for name and address
            name_similarity = jaccard_similarity(name_tokens1, name_tokens2)
            address_similarity = jaccard_similarity(address_tokens1, address_tokens2)
            city_similarity = jaccard_similarity(city_tokens1, city_tokens2)
            # Rule for matching:
            # 1. Jaccard similarity between names is higher than MAX_NAME_DISTANCE
            # 2. Jaccard similarity between addresses is higher than MAX_ADDRESS_DISTANCE
            # 3. Distance between cities is smaller or equal to MAX_CITY_DISTANCE
            if (name_similarity >= MAX_NAME_DISTANCE) and (address_similarity >= MAX_ADDRESS_DISTANCE) and (city_similarity >= MAX_CITY_DISTANCE):
                number_of_matches += 1
                matches.append((df.iloc[i, 0], df.iloc[j, 0]))
    end = time.process_time()
    return matches, number_of_matches, end - start

In [63]:
matches, number_of_matches, time = detect_matches(df_restaurants)

In [65]:
print("Number of matches: {}".format(number_of_matches))
print("Processing time: {}".format(time))

Number of matches: 99
Processing time: 55.856641356


In [66]:
# Display results
for match in matches:
    print("The following records {} and {} match".format(match[0],match[1]))
    print("The restaurants with the following names {} and {} match.".format(df_restaurants.iloc[match[0],1],df_restaurants.iloc[match[1],1]))
    print("The restaurants with the following addresses {} and {} match.".format(df_restaurants.iloc[match[0],2],df_restaurants.iloc[match[1],2]))
    print("\n")

The following records 2 and 753 match
The restaurants with the following names 21 club and 21 club match.
The restaurants with the following addresses  21 w. 52nd st. and  21 w. 52nd st. match.


The following records 6 and 754 match
The restaurants with the following names abruzzi and abruzzi match.
The restaurants with the following addresses  2355 peachtree rd.  peachtree battle shopping center and  2355 peachtree rd. ne match.


The following records 13 and 755 match
The restaurants with the following names alain rondelli and alain rondelli match.
The restaurants with the following addresses  126 clement st. and  126 clement st. match.


The following records 26 and 756 match
The restaurants with the following names aquavit and aquavit match.
The restaurants with the following addresses  13 w. 54th st. and  13 w. 54th st. match.


The following records 27 and 757 match
The restaurants with the following names aqua and aqua match.
The restaurants with the following addresses  252 ca

Note that the rule applied in the above code is not great. You may want to try other kind of distances, other thresholds, and other rules to identify matches.

<span style='color:blue'>
Try to change the code above, use different rules and change the distances used for matching the attributes values. In essence, a good entity resolution program needs to minimize the overall execution time without compromising the quality of the results. Before doing so, we show below how can we assess the results obtained by comparing them to the ground truth results.    
</span>

# Assessing the quality of the results

To do so, we first need to compute the ground truth (that is the list of correct matches) considering the attribute unique_id.

In [67]:
ground_truth_matches = pd.read_csv("restaurants.csv")

In [68]:
ground_truth_matches.insert(0, 'record_ID', range(0, len(ground_truth_matches)))

In [69]:
ground_truth_matches.head(5)

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173


In [70]:
ground_truth_matches = pd.merge(ground_truth_matches,
                                ground_truth_matches,
                                on = 'unique_id')

In [71]:
ground_truth_matches.head(5)

,record_ID_x,name_x,address_x,city_x,cuisine_x,unique_id,record_ID_y,name_y,address_y,city_y,cuisine_y
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675,0,103 west,103 w. paces ferry rd.,atlanta,continental
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172,1,20 mott,20 mott st. between bowery and pell st.,new york,asian
2,2,21 club,21 w. 52nd st.,new york,american,23,2,21 club,21 w. 52nd st.,new york,american
3,2,21 club,21 w. 52nd st.,new york,american,23,753,21 club,21 w. 52nd st.,new york city,american (new)
4,753,21 club,21 w. 52nd st.,new york city,american (new),23,2,21 club,21 w. 52nd st.,new york,american


In [72]:
ground_truth_matches = ground_truth_matches.query('record_ID_x < record_ID_y')

In [73]:
ground_truth_matches.head(5)

,record_ID_x,name_x,address_x,city_x,cuisine_x,unique_id,record_ID_y,name_y,address_y,city_y,cuisine_y
3,2,21 club,21 w. 52nd st.,new york,american,23,753,21 club,21 w. 52nd st.,new york city,american (new)
10,6,abruzzi,2355 peachtree rd. peachtree battle shopping...,atlanta,italian,74,754,abruzzi,2355 peachtree rd. ne,atlanta,italian
20,13,alain rondelli,126 clement st.,san francisco,french,94,755,alain rondelli,126 clement st.,san francisco,french (new)
36,26,aquavit,13 w. 54th st.,new york,continental,24,756,aquavit,13 w. 54th st.,new york city,scandinavian
40,27,aqua,252 california st.,san francisco,seafood,95,757,aqua,252 california st.,san francisco,american (new)


In [74]:
ground_truth_matches = ground_truth_matches[['record_ID_x','record_ID_y']]

In [75]:
print(ground_truth_matches)

      record_ID_x  record_ID_y
3               2          753
10              6          754
20             13          755
36             26          756
40             27          757
...           ...          ...
1030          708          860
1034          709          861
1041          713          862
1053          722          863
1078          744          864

[112 rows x 2 columns]


In [76]:
ground_truth_matches[ground_truth_matches['record_ID_y'] == 753]

,record_ID_x,record_ID_y
3,2,753


In [77]:
ground_truth_matches

,record_ID_x,record_ID_y
3,2,753
10,6,754
20,13,755
36,26,756
40,27,757
...,...,...
1030,708,860
1034,709,861
1041,713,862
1053,722,863


In [78]:
matches_df = pd.DataFrame(matches)
matches_df.columns= ['record_ID_x','record_ID_y']

In [79]:
matches_df.head()

,record_ID_x,record_ID_y
0,2,753
1,6,754
2,13,755
3,26,756
4,27,757


In [80]:
ground_truth_matches.head()

,record_ID_x,record_ID_y
3,2,753
10,6,754
20,13,755
36,26,756
40,27,757


In [81]:
diff_df = pd.merge(ground_truth_matches, matches_df, how='outer', indicator='Exist')

In [82]:
diff_df.head()

,record_ID_x,record_ID_y,Exist
0,2,753,both
1,6,754,both
2,13,755,both
3,26,756,both
4,27,757,both


In [83]:
true_positives = diff_df[diff_df.Exist=='both']
false_positives = diff_df[diff_df.Exist=='right_only']
false_negatives = diff_df[diff_df.Exist=='left_only']

In [84]:
true_positives.head()

,record_ID_x,record_ID_y,Exist
0,2,753,both
1,6,754,both
2,13,755,both
3,26,756,both
4,27,757,both


In [85]:
false_positives.head()

,record_ID_x,record_ID_y,Exist
112,96,579,right_only
113,253,254,right_only
114,463,635,right_only
115,524,525,right_only


In [86]:
false_negatives.sample(5)

,record_ID_x,record_ID_y,Exist
110,722,863,left_only
99,628,853,left_only
100,636,854,left_only
71,465,823,left_only
111,744,864,left_only


In [87]:
df_restaurants

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173
...,...,...,...,...,...,...
860,860,union square cafe,21 e. 16th st.,new york city,american (new),65
861,861,valentino,3115 pico blvd.,santa monica,italian,21
862,862,veni vidi vici,41 14th st.,atlanta,italian,93
863,863,virgils real bbq,152 w. 44th st.,new york city,bbq,66


In [88]:
#Example of a true positive
df_restaurants[df_restaurants.record_ID.isin([2,753])]

,record_ID,name,address,city,cuisine,unique_id
2,2,21 club,21 w. 52nd st.,new york,american,23
753,753,21 club,21 w. 52nd st.,new york city,american (new),23


In [89]:
#Example of a false positive
df_restaurants[df_restaurants.record_ID.isin([463,481])]

,record_ID,name,address,city,cuisine,unique_id
463,463,michaels (las vegas),3595 las vegas blvd. s.,las vegas,continental,664
481,481,mortons of chicago (las vegas),3200 las vegas blvd. s.,las vegas,steakhouses,667


In [90]:
#Example of a false negative
df_restaurants[df_restaurants.record_ID.isin([545,833])]

,record_ID,name,address,city,cuisine,unique_id
545,545,philippes the original,1001 n. alameda st.,los angeles,american,17
833,833,philippe the original,1001 n. alameda st.,chinatown,cafeterias,17


In [91]:
def jaccard_similarity(str1, str2):
    set1 = set(str1.lower().split())
    set2 = set(str2.lower().split())
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union != 0 else 0

str1 = "los angeles"
str2 = "w. hollywood"
similarity = jaccard_similarity(str1, str2)
print("Jaccard Similarity:", similarity)


Jaccard Similarity: 0.0


In [92]:
precision = len(true_positives)/(len(true_positives)+ len(false_positives))
print(precision)

0.9595959595959596


Note that if you are using pyton 2.7 (instead of Python 3), you would need to convert integers to float prior to performing the division

In [93]:
recall = len(true_positives)/(len(true_positives)+ len(false_negatives))
print(recall)

0.8482142857142857


In [94]:
f_measure = 2*(precision*recall)/(precision+recall)
print(f_measure)

0.900473933649289


<span style='color:blue'>
The above results is not a good one, this is due to the rule chosen when decising if two records match or not.
Try to modify the rules, by first examining the dataset, to see if you can improve the precision and recall.
</span>

# Windowing (SNM) method

In this section, you will have to implement the SNM method. It will take as parameters the size of the window, and the attributes based on which the tuples will be ordered.

In [ ]:
from time import process_time
window = 5
df_restaurants = df_restaurants.sort_values(by=['name', 'address'])

number_of_matches = 0
num_records = len(df_restaurants)
matches = []

start = process_time()
for i in range(0,min(window,len(df_restaurants))):
    for j in range(i+1,min(window,len(df_restaurants))):
        # to print the IDs of the records compared, uncomment the following statement
        print("Initial window ({},{})".format(i,j))

        name_tokens1 = df_restaurants.iloc[i, 1]
        name_tokens2 = df_restaurants.iloc[j, 1]
        address_tokens1 = df_restaurants.iloc[i, 2]
        address_tokens2 = df_restaurants.iloc[j, 2]
        city_tokens1 = df_restaurants.iloc[i, 3]
        city_tokens2 = df_restaurants.iloc[j, 3]
            # Calculate Jaccard similarity for name and address
        name_similarity = jaccard_similarity(name_tokens1, name_tokens2)
        address_similarity = jaccard_similarity(address_tokens1, address_tokens2)
        city_similarity = jaccard_similarity(city_tokens1, city_tokens2)
        if (name_similarity >= MAX_NAME_DISTANCE) and (address_similarity >= MAX_ADDRESS_DISTANCE) and (city_similarity >= MAX_CITY_DISTANCE):
                number_of_matches += 1
                matches.append((min(df_restaurants.iloc[j,0],df_restaurants.iloc[i,0]),max(df_restaurants.iloc[j,0],df_restaurants.iloc[i,0])))
for i in range(window,len(df_restaurants)):
    for j in range(i-window+1,i):
        # to print the IDs of the records compared, uncomment the following statement
        print("({},{})".format(j,i))

        name_tokens1 = df_restaurants.iloc[i, 1]
        name_tokens2 = df_restaurants.iloc[j, 1]
        address_tokens1 = df_restaurants.iloc[i, 2]
        address_tokens2 = df_restaurants.iloc[j, 2]
        city_tokens1 = df_restaurants.iloc[i, 3]
        city_tokens2 = df_restaurants.iloc[j, 3]
            # Calculate Jaccard similarity for name and address
        name_similarity = jaccard_similarity(name_tokens1, name_tokens2)
        address_similarity = jaccard_similarity(address_tokens1, address_tokens2)
        city_similarity = jaccard_similarity(city_tokens1, city_tokens2)
        if (name_similarity >= MAX_NAME_DISTANCE) and (address_similarity >= MAX_ADDRESS_DISTANCE) and (city_similarity >= MAX_CITY_DISTANCE):
                number_of_matches += 1
                matches.append((min(df_restaurants.iloc[j,0],df_restaurants.iloc[i,0]),max(df_restaurants.iloc[j,0],df_restaurants.iloc[i,0])))
end = process_time()
print("Number of matches: {}".format(number_of_matches))
print("Processing time: {}".format(end - start))


Initial window (0,1)
Initial window (0,2)
Initial window (0,3)
Initial window (0,4)
Initial window (1,2)
Initial window (1,3)
Initial window (1,4)
Initial window (2,3)
Initial window (2,4)
Initial window (3,4)
(1,5)
(2,5)
(3,5)
(4,5)
(2,6)
(3,6)
(4,6)
(5,6)
(3,7)
(4,7)
(5,7)
(6,7)
(4,8)
(5,8)
(6,8)
(7,8)
(5,9)
(6,9)
(7,9)
(8,9)
(6,10)
(7,10)
(8,10)
(9,10)
(7,11)
(8,11)
(9,11)
(10,11)
(8,12)
(9,12)
(10,12)
(11,12)
(9,13)
(10,13)
(11,13)
(12,13)
(10,14)
(11,14)
(12,14)
(13,14)
(11,15)
(12,15)
(13,15)
(14,15)
(12,16)
(13,16)
(14,16)
(15,16)
(13,17)
(14,17)
(15,17)
(16,17)
(14,18)
(15,18)
(16,18)
(17,18)
(15,19)
(16,19)
(17,19)
(18,19)
(16,20)
(17,20)
(18,20)
(19,20)
(17,21)
(18,21)
(19,21)
(20,21)
(18,22)
(19,22)
(20,22)
(21,22)
(19,23)
(20,23)
(21,23)
(22,23)
(20,24)
(21,24)
(22,24)
(23,24)
(21,25)
(22,25)
(23,25)
(24,25)
(22,26)
(23,26)
(24,26)
(25,26)
(23,27)
(24,27)
(25,27)
(26,27)
(24,28)
(25,28)
(26,28)
(27,28)
(25,29)
(26,29)
(27,29)
(28,29)
(26,30)
(27,30)
(28,30)
(29,30)
(27,31)


(409,411)
(410,411)
(408,412)
(409,412)
(410,412)
(411,412)
(409,413)
(410,413)
(411,413)
(412,413)
(410,414)
(411,414)
(412,414)
(413,414)
(411,415)
(412,415)
(413,415)
(414,415)
(412,416)
(413,416)
(414,416)
(415,416)
(413,417)
(414,417)
(415,417)
(416,417)
(414,418)
(415,418)
(416,418)
(417,418)
(415,419)
(416,419)
(417,419)
(418,419)
(416,420)
(417,420)
(418,420)
(419,420)
(417,421)
(418,421)
(419,421)
(420,421)
(418,422)
(419,422)
(420,422)
(421,422)
(419,423)
(420,423)
(421,423)
(422,423)
(420,424)
(421,424)
(422,424)
(423,424)
(421,425)
(422,425)
(423,425)
(424,425)
(422,426)
(423,426)
(424,426)
(425,426)
(423,427)
(424,427)
(425,427)
(426,427)
(424,428)
(425,428)
(426,428)
(427,428)
(425,429)
(426,429)
(427,429)
(428,429)
(426,430)
(427,430)
(428,430)
(429,430)
(427,431)
(428,431)
(429,431)
(430,431)
(428,432)
(429,432)
(430,432)
(431,432)
(429,433)
(430,433)
(431,433)
(432,433)
(430,434)
(431,434)
(432,434)
(433,434)
(431,435)
(432,435)
(433,435)
(434,435)
(432,436)
(433,436)


(815,819)
(816,819)
(817,819)
(818,819)
(816,820)
(817,820)
(818,820)
(819,820)
(817,821)
(818,821)
(819,821)
(820,821)
(818,822)
(819,822)
(820,822)
(821,822)
(819,823)
(820,823)
(821,823)
(822,823)
(820,824)
(821,824)
(822,824)
(823,824)
(821,825)
(822,825)
(823,825)
(824,825)
(822,826)
(823,826)
(824,826)
(825,826)
(823,827)
(824,827)
(825,827)
(826,827)
(824,828)
(825,828)
(826,828)
(827,828)
(825,829)
(826,829)
(827,829)
(828,829)
(826,830)
(827,830)
(828,830)
(829,830)
(827,831)
(828,831)
(829,831)
(830,831)
(828,832)
(829,832)
(830,832)
(831,832)
(829,833)
(830,833)
(831,833)
(832,833)
(830,834)
(831,834)
(832,834)
(833,834)
(831,835)
(832,835)
(833,835)
(834,835)
(832,836)
(833,836)
(834,836)
(835,836)
(833,837)
(834,837)
(835,837)
(836,837)
(834,838)
(835,838)
(836,838)
(837,838)
(835,839)
(836,839)
(837,839)
(838,839)
(836,840)
(837,840)
(838,840)
(839,840)
(837,841)
(838,841)
(839,841)
(840,841)
(838,842)
(839,842)
(840,842)
(841,842)
(839,843)
(840,843)
(841,843)
(842,843)


In [ ]:
# Display results
for match in matches:
    print("The following records {} and {} match".format(match[0],match[1]))
    print("The restaurants with the following names {} and {} match.".format(df_restaurants.iloc[match[0],1],df_restaurants.iloc[match[1],1]))
    print("The restaurants with the following addresses {} and {} match.".format(df_restaurants.iloc[match[0],2],df_restaurants.iloc[match[1],2]))
    print("\n")

The following records 193 and 784 match
The restaurants with the following names jo jo and mangia e bevi match.
The restaurants with the following addresses  160 e. 64th st. and  800 9th ave. at 53rd st. match.


The following records 76 and 764 match
The restaurants with the following names lauberge and mo better meatty meat match.
The restaurants with the following addresses  1191 1st ave.  between 64th and 65th sts. and  7261 melrose ave. match.


The following records 105 and 769 match
The restaurants with the following names lillie langtrys and tavola calda match.
The restaurants with the following addresses  129 e. fremont st. and  7371 melrose ave. match.


The following records 581 and 845 match
The restaurants with the following names nate n als and brighton coffee shop match.
The restaurants with the following addresses  414 n. beverly dr. and  9600 brighton way match.


The following records 531 and 830 match
The restaurants with the following names fiore rotisserie & grille

In [ ]:
matches_df = pd.DataFrame(matches)
matches_df.columns= ['record_ID_x','record_ID_y']


# reorganization du pair pour avoir l'enregistrement avec le record_id en premier
matches_df['MIN'] = matches_df[['record_ID_x','record_ID_y']].min(axis=1)
matches_df['MAX'] = matches_df[['record_ID_x','record_ID_y']].max(axis=1)
matches_df=matches_df[['MIN','MAX']]
matches_df.columns=['record_ID_x','record_ID_y']

diff_df = pd.merge(ground_truth_matches, matches_df, how='outer', indicator='Exist')
true_positives = diff_df[diff_df.Exist=='both']
false_positives = diff_df[diff_df.Exist=='right_only']
false_negatives = diff_df[diff_df.Exist=='left_only']
precision = len(true_positives)/(len(true_positives)+ len(false_positives))
print(precision)
recall = len(true_positives)/(len(true_positives)+ len(false_negatives))
print(recall)
f_measure = 2*(precision*recall)/(precision+recall)
print(f_measure)

0.978494623655914
0.8125
0.8878048780487805


<span style='color:blue'>
Try to edit the code above, use different keys for sorting, different sizes of windows a,d different rules, and examine the results obtained.
</span>

- sort by name :
    1. window 5
    - 0.978494623655914
    - 0.8125
    - 0.8878048780487805
    2. window 3
    - 0.9891304347826086
    - 0.8125
    - 0.892156862745098
    3. window 8
    - 0.978494623655914
    - 0.8125
    - 0.8878048780487805
    4. window 10
    - 0.978494623655914
    - 0.8125
    - 0.8878048780487805


    
- sort by address :
    1. window 3
    - 1.0
    - 0.7857142857142857
    - 0.88
    2. window 5
    - 0.989247311827957
    - 0.8214285714285714
    - 0.8975609756097561
    3. window 8
    - 0.989247311827957
    - 0.8214285714285714
    - 0.8975609756097561


- sort by city :
    - 0.9743589743589743
    - 0.3392857142857143
    - 0.5033112582781457
    
- sort by cuisine :
    - 0.9166666666666666
    - 0.09821428571428571
    - 0.1774193548387097
    
- sort by name and address :
    - 0.978494623655914
    - 0.8125
    - 0.8878048780487805

The analysis of the results obtained from testing different keys for ordering and window sizes reveals valuable insights into the effectiveness of each approach for matching tuples within a dataset. Sorting by name consistently provides stable and reliable results, with high precision and recall regardless of the window size used. Sorting by address emerges as the most effective approach, yielding perfect precision and maintaining high recall across various window sizes. In contrast, sorting by city and cuisine leads to lower recall, indicating a higher rate of missed true matches. These findings suggest that sorting by address offers the most robust method for matching tuples based on their attributes, while sorting by name also proves to be a reliable alternative.

We can explain those results by the fact that :
- Sorting By City offers an uneffective approach, if we were to take the city of new york, we would have multitudes of restaurants that may not be the same, therefore complexifying the algorithm

It is worth noting that in the above code, we do not implement the SNM algorithm in its entirety. In particular, we do not implement the last phase of inferring matches using transitivity

# Blocking method

<span style='color:blue'>
Now that we have seen how the exhaustive and windowing methods work, can you try to examine how the blocking method can be implemented. In oparticular, can you spacify how the blocks can be formed; To do so, examine the restaurant dataset.
</span>

In [124]:
import time
import pandas as pd


# Function to create a blocking key based on the first letters of the name and address
def create_blocking_key(name, address, city):
    return (name[:2].lower(), address[:2].lower(), city[:2].lower())

# Function to implement blocking
def block_and_detect_matches(df):
    # Create blocks based on the first letters of the name and address
    df['block_key'] = df.apply(lambda row: create_blocking_key(row['name'], row['address'], row['city']), axis=1)
    blocks = df.groupby('block_key')
    all_matches = []
    total_matches = 0
    total_time = 0

    # Apply duplicate detection within each block
    for block_key, block in blocks:
        matches, number_of_matches, elapsed_time = detect_matches(block)
        all_matches.extend(matches)
        total_matches += number_of_matches
        total_time += elapsed_time

    return all_matches, total_matches, total_time



# Apply the blocking method

matches, number_of_matches, elapsed_time = block_and_detect_matches(df_restaurants)

# Print results
print("Matches:", matches)
print("Number of Matches:", number_of_matches)
print("Elapsed Time:", elapsed_time)


Matches: [(2, 753), (6, 754), (13, 755), (26, 756), (27, 757), (30, 758), (36, 760), (37, 761), (76, 764), (73, 763), (79, 765), (92, 766), (105, 769), (99, 767), (109, 770), (141, 773), (132, 772), (130, 771), (103, 768), (156, 775), (163, 777), (164, 778), (167, 779), (170, 780), (186, 782), (190, 783), (193, 784), (233, 785), (245, 787), (250, 788), (257, 789), (264, 790), (274, 791), (278, 793), (275, 792), (294, 795), (296, 796), (297, 797), (321, 798), (326, 799), (339, 800), (358, 802), (375, 806), (371, 805), (367, 803), (369, 804), (385, 807), (396, 809), (417, 814), (415, 813), (450, 820), (446, 818), (438, 817), (434, 816), (448, 819), (461, 822), (477, 824), (497, 825), (504, 826), (529, 829), (519, 827), (534, 831), (524, 525), (544, 832), (548, 834), (553, 836), (556, 837), (562, 838), (567, 839), (577, 840), (581, 845), (841, 842), (589, 846), (605, 847), (616, 848), (619, 850), (617, 849), (625, 852), (643, 855), (687, 857), (708, 860), (709, 861), (713, 862)]
Number of

In [114]:
pip install metaphone

  Preparing metadata (setup.py) ... done
  Created wheel for metaphone: filename=Metaphone-0.6-py3-none-any.whl size=13902 sha256=5b1c6c85d9eb50fa6cccc926cbe46d68028045cb9b45c94a3e78765a83d3c064
  Stored in directory: /root/.cache/pip/wheels/23/dd/1d/6cdd346605db62bde1f60954155e9ce48f4681c243f265b704
Successfully built metaphone


In [120]:
import time
import pandas as pd
from nltk.corpus import stopwords
from nltk.metrics.distance import edit_distance
from nltk.tokenize import word_tokenize
from metaphone import doublemetaphone

nltk.download('punkt')

# Function to apply double metaphone and generate phonetic keys
def metaphone_key(text):
    return doublemetaphone(text)[0]

# Function to create a blocking key based on phonetic keys of name and address
def create_blocking_key(name, address, city):
    name_key = metaphone_key(name)
    address_key = metaphone_key(address)
    city_key = metaphone_key(city)
    return (name_key[:2], address_key[:2], city_key[:2])

# Function to implement blocking
def block_and_detect_matches(df):
    # Create blocks based on phonetic keys of the name, address, and city
    df['block_key'] = df.apply(lambda row: create_blocking_key(row['name'], row['address'], row['city']), axis=1)
    blocks = df.groupby('block_key')
    all_matches = []
    total_matches = 0
    total_time = 0

    # Apply duplicate detection within each block
    for block_key, block in blocks:
        matches, number_of_matches, elapsed_time = detect_matches(block)
        all_matches.extend(matches)
        total_matches += number_of_matches
        total_time += elapsed_time

    return all_matches, total_matches, total_time


# Apply the blocking method
matches, number_of_matches, elapsed_time = block_and_detect_matches(df_restaurants)

# Print results
print("Matches:", matches)
print("Number of Matches:", number_of_matches)
print("Elapsed Time:", elapsed_time)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Matches: [(26, 756), (27, 757), (326, 799), (13, 755), (708, 860), (321, 798), (6, 754), (30, 758), (36, 760), (504, 826), (233, 785), (709, 861), (245, 787), (713, 862), (250, 788), (294, 795), (296, 796), (297, 797), (339, 800), (264, 790), (105, 769), (99, 767), (103, 768), (109, 770), (2, 753), (130, 771), (132, 772), (358, 802), (275, 792), (278, 793), (141, 773), (274, 791), (371, 805), (367, 803), (369, 804), (375, 806), (415, 813), (396, 809), (385, 807), (417, 814), (461, 822), (434, 816), (477, 824), (438, 817), (446, 818), (448, 819), (450, 820), (497, 825), (92, 766), (556, 837), (519, 827), (76, 764), (524, 525), (529, 829), (73, 763), (79, 765), (562, 838), (544, 832), (534, 831), (548, 834), (37, 761), (581, 845), (577, 840), (567, 839), (589, 846), (579, 841), (841, 842), (617, 849), (605, 847), (167, 779), (619, 850), (643, 855), (170, 780), (687, 857), (193, 784), (693, 858), (186, 782), (190, 783), (624, 851), (164, 778), (163, 777), (156, 775)]
Number of Matches: 82

In [125]:
matches_df = pd.DataFrame(matches)
matches_df.columns= ['record_ID_x','record_ID_y']
matches_df['record_ID_x'] = matches_df['record_ID_x'].astype(int)
matches_df['record_ID_y'] = matches_df['record_ID_y'].astype(int)




# reorganization du pair pour avoir l'enregistrement avec le record_id en premier
matches_df['MIN'] = matches_df[['record_ID_x','record_ID_y']].min(axis=1)
matches_df['MAX'] = matches_df[['record_ID_x','record_ID_y']].max(axis=1)
matches_df=matches_df[['MIN','MAX']]
matches_df.columns=['record_ID_x','record_ID_y']

diff_df = pd.merge(ground_truth_matches, matches_df, how='outer', indicator='Exist')
true_positives = diff_df[diff_df.Exist=='both']
false_positives = diff_df[diff_df.Exist=='right_only']
false_negatives = diff_df[diff_df.Exist=='left_only']
precision = len(true_positives)/(len(true_positives)+ len(false_positives))
print(precision)
recall = len(true_positives)/(len(true_positives)+ len(false_negatives))
print(recall)
f_measure = 2*(precision*recall)/(precision+recall)
print(f_measure)

0.9759036144578314
0.7232142857142857
0.8307692307692307
